# Logic, in Products

Companion notebook for the [*Logic, in Products*](https://Notabot123.github.io/polyweave/blog/differentiable-logic/) blog post.
Run the cells top to bottom. Everything uses `polyweave` + `torch`.


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from polyweave import PolyLinear
from polyweave.logic import fuzzy_and, fuzzy_or, fuzzy_xor, SoftSignedLiteral, SoftRuleLayer
from polyweave.reasoning import PropKB, ForwardChainer

torch.manual_seed(0)

## Conjunction is multiplication

With the product t-norm, a fuzzy AND is literally a product neuron. The gates are exact on
the Boolean corners and differentiable in between.

In [ ]:
for a in (0.0, 1.0):
    for b in (0.0, 1.0):
        ta, tb = torch.tensor(a), torch.tensor(b)
        print(int(a), int(b), "|",
              f"AND={fuzzy_and(ta, tb):.0f}",
              f"OR={fuzzy_or(ta, tb):.0f}",
              f"XOR={fuzzy_xor(ta, tb):.0f}")

## XOR in a single neuron

`fuzzy_xor` is `a + b - 2ab` — a linear term **plus a product**. A single rank-1
`PolyLinear` has exactly that bilinear term, so it learns XOR where `nn.Linear` cannot.

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

def fit(model, steps=4000, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        F.mse_loss(model(X), Y).backward()
        opt.step()
    preds = [round(v, 2) for v in model(X).detach().squeeze().tolist()]
    return round(F.mse_loss(model(X), Y).item(), 3), preds

print("PolyLinear(2,1,rank=1):", fit(PolyLinear(2, 1, rank=1)))
print("nn.Linear(2,1):        ", fit(torch.nn.Linear(2, 1)))

## Reasoning as a differentiable layer

Scale conjunction up and you get inference: forward chaining fires rules (product-AND of
premises) to a fixpoint. It's differentiable in the facts, so it's a reasoning *layer*.

In [ ]:
kb = PropKB()
kb.add_rule(["raining"], "wet_grass")
kb.add_rule(["wet_grass"], "slippery")
kb.add_rule(["wet_grass", "sunny"], "rainbow")   # a conjunction

chainer = ForwardChainer(kb)
print("entails slippery:", chainer.entails(kb.initial_facts(["raining"]), "slippery"))
print("entails rainbow :", chainer.entails(kb.initial_facts(["raining"]), "rainbow"))  # needs sunny

And `plot_chaining_trace` shows the truth values propagating to the fixpoint - each row is a chaining step, and a cell turns green as its fact becomes true.

In [ ]:
from polyweave.viz import plot_chaining_trace
from IPython.display import Image

_, history = chainer(kb.initial_facts(["raining"]), return_history=True)
paths = plot_chaining_trace(history, kb.fact_names, "chaining_trace")
Image(str(next(p for p in paths if str(p).endswith(".png"))))

## Inducing rules — with negation

Attach one **signed exponent** per premise and the product becomes a rule body the
optimiser *induces* — including negation (`w<0` = an inhibitory / negated literal).
`SoftSignedLiteral` learns `fly = bird AND NOT penguin` and you read it off the exponents.

In [ ]:
rng = torch.Generator().manual_seed(0)
feats = ["bird", "penguin", "distractor_a", "distractor_b"]
Xs = (torch.rand(4000, 4, generator=rng) < 0.5).float()
ys = (Xs[:, 0] * (1 - Xs[:, 1])).unsqueeze(1)        # bird AND NOT penguin

lit = SoftSignedLiteral(4, signed=True)
opt = torch.optim.Adam(lit.parameters(), lr=0.05)
for _ in range(300):
    opt.zero_grad()
    F.binary_cross_entropy(lit(Xs).clamp(1e-6, 1 - 1e-6), ys).backward()
    opt.step()

for name, role, w in lit.literals(feats):
    print(f"{name:<12} {role:<11} w={w:+.2f}")

PolyWeave's `viz` module renders this for you with `plot_rule_exponents` - required literals are green, inhibitory (negated) ones vermillion. It saves a figure to `plots/` and returns the paths, so we display the PNG inline.

In [ ]:
from polyweave.viz import plot_rule_exponents
from IPython.display import Image

paths = plot_rule_exponents(
    {"fly = bird & not penguin": dict(zip(feats, lit.w.tolist()))},
    "rule_exponents",
)
Image(str(next(p for p in paths if str(p).endswith(".png"))))

`SoftRuleLayer` ORs several literals into a soft DNF, so it can induce a multi-rule,
non-linearly-separable theory — and recover both rules with their negations.

In [ ]:
rng2 = torch.Generator().manual_seed(1)
fd = ["bird", "penguin", "bat", "broken"]
Xd = (torch.rand(6000, 4, generator=rng2) < 0.5).float()
t1 = Xd[:, 0] * (1 - Xd[:, 1])      # bird AND NOT penguin
t2 = Xd[:, 2] * (1 - Xd[:, 3])      # bat  AND NOT broken
yd = (1 - (1 - t1) * (1 - t2)).unsqueeze(1)          # OR

rules = SoftRuleLayer(4, n_rules=2)
opt = torch.optim.Adam(rules.parameters(), lr=0.05)
for _ in range(300):
    opt.zero_grad()
    F.binary_cross_entropy(rules(Xd).clamp(1e-6, 1 - 1e-6), yd).backward()
    opt.step()

print("induced DNF:")
for r in rules.rules_text(fd):
    print("  -", r)

That's the toolkit: fuzzy gates, XOR in one neuron, differentiable forward chaining, and
interpretable rule induction with negation — all on PolyWeave's multiplicative core.